In [1]:
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
df = pd.read_csv("news_data.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(19813, 7)
['title', 'author', 'publish_date', 'article_text', 'url', 'main_image', 'tag']


,title,author,publish_date,article_text,url,main_image,tag
0,Hasto Ingin Debat Pilpres Pertarungan Gagasan ...,detikNews,2023-12-10 23:31:00+07:00,Sekjen PDIP sekaligus Sekretaris Tim Pemenang ...,https://news.detik.com/pemilu/d-7082492/hasto-...,https://awsimages.detik.net.id/community/media...,"['hasto kristiyanto', 'hasto', 'sekjen pdip', ..."
1,"Terima Kunjungan Gibran di Ponpes Al-Tsaqafah,...",detikNews,2023-12-10 23:09:00+07:00,Mantan Ketua Umum Pengurus Besar Nahdlatul Ula...,https://news.detik.com/pemilu/d-7082483/terima...,NaN,"['said aqil siradj', 'gibran rakabuming', 'pem..."
2,Contoh Format dan Isi Surat Pernyataan Anggota...,detikNews,2023-12-10 22:38:00+07:00,Surat pernyataan KPPS Pemilu 2024 merupakan ba...,https://news.detik.com/pemilu/d-7082424/contoh...,https://awsimages.detik.net.id/community/media...,"['surat pernyataan kpps pemilu 2024', 'pendaft..."
3,9 Ribu Personel Gabungan Dikerahkan Amankan Na...,detikSumut,2023-12-10 22:23:00+07:00,Sekitar 9.000 personel dikerahkan untuk mengam...,https://www.detik.com/sumut/berita/d-7082448/9...,https://awsimages.detik.net.id/community/media...,"['natal', 'tahun baru', 'nataru', 'pengamanan ..."
4,Amarah Ukraina Usai Rusia Bakal Gelar Pilpres ...,detikNews,2023-12-10 22:12:00+07:00,Ukraina marah usai Rusia menjadikan empat wila...,https://news.detik.com/internasional/d-7082454...,https://awsimages.detik.net.id/community/media...,"['ukraina', 'rusia', 'pemilu rusia', 'round-up']"


In [3]:
# Pilih kolom teks otomatis (kolom teks dengan rata-rata panjang terbesar).
# Jika salah, ganti manual, misal: text_column = "content"
kandidat = df.select_dtypes(include="object").columns
text_column = max(kandidat, key=lambda c: df[c].astype(str).str.len().mean())
print("Kolom teks yang dipakai:", text_column)

df = df.dropna(subset=[text_column]).reset_index(drop=True)

# Stemming Sastrawi cukup lambat. Untuk data besar, batasi dulu jumlah baris
# (hapus/ubah baris di bawah untuk memproses seluruh dataset).
# df = df.head(2000).copy()
print("Jumlah dokumen:", len(df))

Kolom teks yang dipakai: article_text
Jumlah dokumen: 19812


C:\Users\ASUS\AppData\Local\Temp\ipykernel_10188\447506501.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  kandidat = df.select_dtypes(include="object").columns


In [4]:
def case_folding(text):
    return str(text).lower()

text = "Saya Belajar NLP"
print(case_folding(text))

saya belajar nlp


In [5]:
def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

text = "Kunjungi https://example.com sekarang"
print(remove_url(text))

Kunjungi  sekarang


In [6]:
def remove_mention(text):
    return re.sub(r"@\w+", "", text)

def remove_hashtag(text):
    return re.sub(r"#\w+", "", text)

print(remove_mention("@user1 produk ini bagus"))
print(remove_hashtag("Wisata #Lombok sangat indah #Pariwisata"))

 produk ini bagus
Wisata  sangat indah 


In [7]:
def remove_number(text):
    return re.sub(r"\d+", "", text)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

print(remove_number("Harga Rp100.000 untuk 5 buah"))
print(remove_punctuation("Bagus banget!!! Mantap, kan?"))
print(repr(clean_whitespace("  banyak    spasi \n\n di sini  ")))

Harga Rp. untuk  buah
Bagus banget Mantap kan
'banyak spasi di sini'


In [8]:
def clean_text(text):
    text = str(text)
    text = text.lower()              # case folding
    text = remove_url(text)          # hapus URL
    text = remove_mention(text)      # hapus mention
    text = remove_hashtag(text)      # hapus hashtag
    text = remove_number(text)       # hapus angka
    text = remove_punctuation(text)  # hapus punctuation
    text = clean_whitespace(text)    # bersihkan whitespace
    return text

df["clean_text"] = df[text_column].apply(clean_text)
df[[text_column, "clean_text"]].head(10)

,article_text,clean_text
0,Sekjen PDIP sekaligus Sekretaris Tim Pemenang ...,sekjen pdip sekaligus sekretaris tim pemenang ...
1,Mantan Ketua Umum Pengurus Besar Nahdlatul Ula...,mantan ketua umum pengurus besar nahdlatul ula...
2,Surat pernyataan KPPS Pemilu 2024 merupakan ba...,surat pernyataan kpps pemilu merupakan bagian ...
3,Sekitar 9.000 personel dikerahkan untuk mengam...,sekitar personel dikerahkan untuk mengamankan ...
4,Ukraina marah usai Rusia menjadikan empat wila...,ukraina marah usai rusia menjadikan empat wila...
5,Sekretaris Tim Kampanye Nasional (TKN) Prabowo...,sekretaris tim kampanye nasional tkn prabowo g...
6,Cawapres nomor urut 2 Gibran Rakabuming Raka m...,cawapres nomor urut gibran rakabuming raka men...
7,Sekretaris Tim Pemenangan Nasional (TPN) capre...,sekretaris tim pemenangan nasional tpn capres ...
8,Tim Kampanye Nasional (TKN) menegaskan kapasit...,tim kampanye nasional tkn menegaskan kapasitas...
9,Pasangan capres dan cawapres nomor urut 2 Prab...,pasangan capres dan cawapres nomor urut prabow...


In [9]:
stop_words = set(stopwords.words("indonesian"))
print("Jumlah stopword:", len(stop_words))
print("'tidak' termasuk stopword NLTK?", "tidak" in stop_words)

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

tokens = ["saya", "sedang", "belajar", "nlp", "yang", "menarik"]
print(remove_stopwords(tokens))

# Demonstrasi dampak penghapusan stopword pada kalimat bernegasi
kalimat = "tidak bagus"
tokens = kalimat.split()
print("Sebelum:", tokens)
print("Sesudah:", remove_stopwords(tokens))

Jumlah stopword: 757
'tidak' termasuk stopword NLTK? True
['belajar', 'nlp', 'menarik']
Sebelum: ['tidak', 'bagus']
Sesudah: ['bagus']


In [10]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Cache agar kata yang sama tidak di-stem berulang (jauh lebih cepat pada dataset besar)
stem_cache = {}

def stem_word(token):
    if token not in stem_cache:
        stem_cache[token] = stemmer.stem(token)
    return stem_cache[token]

def stem_tokens(tokens):
    return [stem_word(token) for token in tokens]

print(stemmer.stem("Mahasiswa sedang mempelajari pemrograman."))
for kata in ["mempelajari", "berlari-lari", "ketidakhadiran", "pelayanannya"]:
    print(kata, "->", stemmer.stem(kata))

mahasiswa sedang ajar pemrograman
mempelajari -> ajar
berlari-lari -> lari
ketidakhadiran -> ketidakhadiran
pelayanannya -> layan


In [ ]:
import re

# Fungsi-fungsi pendukungnya dulu
def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_mention(text):
    return re.sub(r'@\w+', '', text)

def remove_hashtag(text):
    return re.sub(r'#\w+', '', text)

def remove_number(text):
    return re.sub(r'\d+', '', text)

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

def clean_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

# Fungsi Cleaning Lengkap
def clean_text(text):
    text = str(text)                      # Ubah jadi teks
    text = text.lower()                   # Ubah jadi huruf kecil
    text = remove_url(text)               # Hapus tautan
    text = remove_mention(text)           # Hapus sebutan
    text = remove_hashtag(text)           # Hapus tagar
    text = remove_number(text)            # Hapus angka
    text = remove_punctuation(text)       # Hapus tanda baca
    text = clean_whitespace(text)          # Rapikan spasi
    return text

# Terapkan ke dataset
text_column = "content"  # sesuaikan nama kolom kamu
df["clean_text"] = df[text_column].apply(clean_text)

# Tampilkan hasil
df[[text_column, "clean_text"]].head()

In [2]:
slang_dict = {
    "gue": "saya", "gw": "saya", "lu": "kamu", "loe": "kamu",
    "bgt": "banget", "yg": "yang", "dgn": "dengan", "utk": "untuk",
    "krn": "karena", "tdk": "tidak", "gak": "tidak", "nggak": "tidak",
    "ga": "tidak", "udah": "sudah", "aja": "saja",
}

def normalize_slang(text):
    return " ".join(slang_dict.get(w, w) for w in text.split())

contoh = "gue udah beli yg bagus bgt tapi gak murah"
print("Sebelum:", contoh)
print("Sesudah:", normalize_slang(contoh))

Sebelum: gue udah beli yg bagus bgt tapi gak murah
Sesudah: saya sudah beli yang bagus banget tapi tidak murah


In [3]:
# Langkah 1: Import library
import pandas as pd
import re
import string

# Langkah 2: Definisi fungsi
def case_folding(text):
    return str(text).lower()

def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

def remove_mention(text):
    return re.sub(r"@\w+", "", text)

def remove_hashtag(text):
    return re.sub(r"#\w+", "", text)

def remove_number(text):
    return re.sub(r"\d+", "", text)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def clean_text(text):
    text = str(text)
    text = case_folding(text)
    text = remove_url(text)
    text = remove_mention(text)
    text = remove_hashtag(text)
    text = remove_number(text)
    text = remove_punctuation(text)
    text = clean_whitespace(text)
    return text

# Langkah 3: Cek file dan baca dataset
import os
print(" File di folder ini:")
print(os.listdir("."))

df = pd.read_csv("news_data.csv")
print("\n Dataset berhasil dibaca!")
print("Kolom yang ada:", df.columns.tolist())

 File di folder ini:
['.venv', 'aldaenv', 'ALDASYAFITRINLP.vnv.ipynb', 'cth1.ipynb', 'dataset_pemilu', 'eksplorasi_corpus.ipynb.ipynb', 'kaggle_pemilu.zip', 'news_data.csv', 'NLP', 'Pertemuan_02_Tekonization.ipynb', 'Pertemuan_03_Normalization.ipynb', 'test_environment.ipynb']

 Dataset berhasil dibaca!
Kolom yang ada: ['title', 'author', 'publish_date', 'article_text', 'url', 'main_image', 'tag']


In [4]:
# Pastikan fungsi-fungsi ini sudah didefinisikan di atas
def case_folding(text):
    return str(text).lower()

def remove_url(text):
    return re.sub(r"http\S+|www\S+", "", text)

def remove_mention(text):
    return re.sub(r"@\w+", "", text)

def remove_hashtag(text):
    return re.sub(r"#\w+", "", text)

def remove_number(text):
    return re.sub(r"\d+", "", text)

def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

# Fungsi Cleaning Lengkap — ini yang ada di foto kamu
def clean_text(text):
    text = str(text)
    text = case_folding(text)
    text = remove_url(text)
    text = remove_mention(text)
    text = remove_hashtag(text)
    text = remove_number(text)
    text = remove_punctuation(text)
    text = clean_whitespace(text)
    return text

# Terapkan ke dataset
text_column = "article_text"  # Nama kolom yang benar
df["clean_text"] = df[text_column].apply(clean_text)
df[[text_column, "clean_text"]].head()

,article_text,clean_text
0,Sekjen PDIP sekaligus Sekretaris Tim Pemenang ...,sekjen pdip sekaligus sekretaris tim pemenang ...
1,Mantan Ketua Umum Pengurus Besar Nahdlatul Ula...,mantan ketua umum pengurus besar nahdlatul ula...
2,Surat pernyataan KPPS Pemilu 2024 merupakan ba...,surat pernyataan kpps pemilu merupakan bagian ...
3,Sekitar 9.000 personel dikerahkan untuk mengam...,sekitar personel dikerahkan untuk mengamankan ...
4,Ukraina marah usai Rusia menjadikan empat wila...,ukraina marah usai rusia menjadikan empat wila...


In [5]:
def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
# Daftar stopword Bahasa Indonesia
stop_words = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "adalah",
    "ini", "itu", "atau", "serta", "pada", "dalam", "untuk", "dengan",
    "akan", "juga", "sudah", "belum", "bisa", "saat", "oleh", "sebagai",
    "harus", "dapat", "kepada", "melalui", "tentang", "seperti", "hingga",
    "saya", "kamu", "dia", "mereka", "kita", "anda", "beliau",
    "tidak", "bukan", "hanya", "saja", "lebih", "kurang", "sangat",
    "namun", "tetapi", "lalu", "kemudian", "lalu", "sementara",
    "bagi", "antara", "setelah", "sebelum", "ketika", "karena", "sebab"
}

# Fungsi hapus stopword
def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

print("Stopword siap digunakan!")

Stopword siap digunakan!


In [9]:
# Contoh mencoba fungsi remove_stopwords
from nltk.tokenize import word_tokenize

teks_uji = ["saya", "sedang", "belajar", "nlp", "yang", "menarik"]
hasil = remove_stopwords(teks_uji)

print("Token asli:", teks_uji)
print("Setelah hapus stopword:", hasil)

Token asli: ['saya', 'sedang', 'belajar', 'nlp', 'yang', 'menarik']
Setelah hapus stopword: ['sedang', 'belajar', 'nlp', 'menarik']


In [11]:
# Langkah 1: Import 
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Langkah 2: Buat stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Langkah 3: Fungsi stemming
def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

# Contoh
print(stemmer.stem("pemerintahan"))
print(stemmer.stem("berlarian"))

perintah
lari


In [22]:
# Terapkan ke 100 data
df_sample = df.head(100).copy()

df_sample["processed_tokens"] = df_sample[text_column].apply(preprocessing_pipeline)
df_sample["processed_text"] = df_sample["processed_tokens"].apply(lambda tokens: " ".join(tokens))

# Tampilkan hasil
print("Selesai! Memproses 100 data")
df_sample[[text_column, "clean_text", "processed_text"]].head(10)

Selesai! Memproses 100 data


,article_text,clean_text,processed_text
0,Sekjen PDIP sekaligus Sekretaris Tim Pemenang ...,sekjen pdip sekaligus sekretaris tim pemenang ...,sekjen pdip sekaligus sekretaris tim menang na...
1,Mantan Ketua Umum Pengurus Besar Nahdlatul Ula...,mantan ketua umum pengurus besar nahdlatul ula...,mantan ketua umum urus besar nahdlatul ulama p...
2,Surat pernyataan KPPS Pemilu 2024 merupakan ba...,surat pernyataan kpps pemilu merupakan bagian ...,surat nyata kpps milu rupa bagi berkas dokumen...
3,Sekitar 9.000 personel dikerahkan untuk mengam...,sekitar personel dikerahkan untuk mengamankan ...,sekitar personel kerah aman raya natal tahun b...
4,Ukraina marah usai Rusia menjadikan empat wila...,ukraina marah usai rusia menjadikan empat wila...,ukraina marah usai rusia jadi empat wilayah ki...
5,Sekretaris Tim Kampanye Nasional (TKN) Prabowo...,sekretaris tim kampanye nasional tkn prabowo g...,sekretaris tim kampanye nasional tkn prabowo g...
6,Cawapres nomor urut 2 Gibran Rakabuming Raka m...,cawapres nomor urut gibran rakabuming raka men...,cawapres nomor urut gibran rakabuming raka dad...
7,Sekretaris Tim Pemenangan Nasional (TPN) capre...,sekretaris tim pemenangan nasional tpn capres ...,sekretaris tim menang nasional tpn capres cawa...
8,Tim Kampanye Nasional (TKN) menegaskan kapasit...,tim kampanye nasional tkn menegaskan kapasitas...,tim kampanye nasional tkn tegas kapasitas prab...
9,Pasangan capres dan cawapres nomor urut 2 Prab...,pasangan capres dan cawapres nomor urut prabow...,pasang capres cawapres nomor urut prabowo subi...


In [19]:
df.to_csv("dataset_preprocessed.csv", index=False)
print(" Dataset berhasil disimpan sebagai dataset_preprocessed.csv")

 Dataset berhasil disimpan sebagai dataset_preprocessed.csv
